In [ ]:
from scapy.all import sniff

# Список сигнатур (шаблонов) для поиска XSS
XSS_SIGNATURES = [
    "<script>",
    "alert(",
    "onerror=",
    "javascript:",
    "prompt(",
    "confirm("
]


def analyze_packet(packet) -> None:
    """Анализирует содержимое сетевого пакета на наличие XSS-сигнатур."""
    # Проверяем, содержит ли пакет слой "Raw" с полезными данными
    if not packet.haslayer("Raw"):
        return

    try:
        # Извлекаем сырые байты и декодируем их в текстовую строку
        raw_data = packet["Raw"].load.decode(encoding="utf-8", errors="ignore")
    except Exception:
        return

    # Приводим текст к нижнему регистру для регистронезависимого поиска
    lower_data = raw_data.lower()

    # Проверяем наличие каждой сигнатуры в пакете
    for signature in XSS_SIGNATURES:
        if signature in lower_data:
            print(f"\n[!] ALERT: Possible XSS Attack Detected!")
            print(f"[-] Triggered by signature: '{signature}'")
            print(f"[-] Payload chunk:\n{raw_data[:500]}\n" + "-" * 50)
            break  # Выходим из цикла, чтобы не спамить, если совпало несколько сигнатур


def main() -> None:
    """Основная функция для запуска сниффера."""
    print("[*] Starting XSS detection engine...")
    print("[*] Listening on TCP port 80 (HTTP)... Press Ctrl+C to stop.")

    try:
        # Запуск перехвата пакетов
        sniff(
            filter="tcp port 80",
            prn=analyze_packet,
            store=0
        )
    except PermissionError:
        print("\n[ERROR] Permission denied. Please run this script with sudo / root privileges.")
    except KeyboardInterrupt:
        print("\n[*] Stopping XSS detection. Goodbye!")


if __name__ == "__main__":
    main()
